# Classification Metrics in Machine Learning

When you train a classifier, accuracy alone doesn't tell the whole story, especially on imbalanced datasets. This notebook covers the five most important classification metrics:

1. **Confusion Matrix** - the foundation everything else is built on
2. **Accuracy** - overall correctness
3. **Precision** - how trustworthy positive predictions are
4. **Recall** - how many actual positives were caught
5. **F1 Score** - balance between precision and recall

We'll use the Iris dataset with a Logistic Regression classifier to demonstrate each one.

In [1]:
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## Dataset & Model Setup

The Iris dataset has 150 samples across 3 classes (Setosa, Versicolor, Virginica). We split it 75/25 into train and test sets, then fit a Logistic Regression classifier.

In [2]:
iris = datasets.load_iris()

x_train, x_test, y_train, y_test = train_test_split(
    iris.data, iris.target, random_state=1
)

clf = LogisticRegression(max_iter=200)
clf.fit(x_train, y_train)

y_train_pred = clf.predict(x_train)
y_test_pred  = clf.predict(x_test)

---
## 1. Confusion Matrix

A confusion matrix lays out every combination of actual vs. predicted class in a grid. For a binary classifier the four cells are:

| | Predicted Positive | Predicted Negative |
|---|---|---|
| **Actual Positive** | True Positive (TP) | False Negative (FN) |
| **Actual Negative** | False Positive (FP) | True Negative (TN) |

- **TP** - model correctly predicted positive  
- **TN** - model correctly predicted negative  
- **FP** - model predicted positive, but it was actually negative (Type I error)  
- **FN** - model predicted negative, but it was actually positive (Type II error)  

For multi-class problems (like Iris with 3 classes), the matrix extends to N x N where each row is the actual class and each column is the predicted class. A perfect classifier has all values on the diagonal.

In [ ]:
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=iris.target_names,
            yticklabels=iris.target_names)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.title('Confusion Matrix - Test Set')
plt.tight_layout()
plt.show()

print(cm)

---
## 2. Accuracy

Accuracy is the simplest metric - the fraction of all predictions that were correct.

$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}$$

**When to use it:** Only reliable when classes are roughly balanced. A model that predicts the majority class every time on a 95/5 split achieves 95% accuracy without learning anything useful.

**When to avoid it:** Imbalanced datasets (fraud detection, disease screening).

In [4]:
acc = accuracy_score(y_test, y_test_pred)
print(f"Test Accuracy: {acc:.4f}  ({acc*100:.2f}%)")

---
## 3. Precision

Precision answers: *"Of all the samples the model labeled as positive, how many actually were positive?"*

$$\text{Precision} = \frac{TP}{TP + FP}$$

**High precision matters when** false positives are costly — e.g., spam filtering (you don't want to delete real emails), or medical tests that lead to invasive procedures.

A model that is very conservative and only predicts positive when it's very sure will have high precision, but may miss many actual positives.

---
## 4. Recall (Sensitivity)

Recall answers: *"Of all the actual positives, how many did the model find?"*

$$\text{Recall} = \frac{TP}{TP + FN}$$

Also called **sensitivity** or **true positive rate**.

**High recall matters when** false negatives are costly — e.g., cancer screening (you really don't want to miss a positive case), fraud detection (missing a fraudulent transaction is worse than a false alarm).

A model that predicts positive for everything will have 100% recall but terrible precision.

---
## 5. F1 Score

Precision and recall trade off against each other. The F1 score is their **harmonic mean**, giving a single number that balances both.

$$\text{F1} = 2 \times \frac{\text{Precision} \times \text{Recall}}{\text{Precision} + \text{Recall}}$$

The harmonic mean penalises extreme imbalances - if either precision or recall is very low, F1 will be low too. A perfect F1 score is 1.0, and the worst is 0.0.

**When to use it:** Whenever you need a single metric that respects both false positives and false negatives, especially on imbalanced datasets.

### Precision-Recall Trade-off Summary

| Scenario | Prioritise |
|---|---|
| Spam filter | Precision (avoid deleting real emails) |
| Cancer screening | Recall (avoid missing cases) |
| General imbalanced data | F1 Score |

In [5]:
# classification_report prints precision, recall, and F1 for every class
print(classification_report(y_test, y_test_pred, target_names=iris.target_names))

### Reading the Report

- **precision** - per-class precision
- **recall** - per-class recall  
- **f1-score** - per-class F1
- **support** - number of actual samples in that class
- **macro avg** - unweighted mean across classes (treats all classes equally)
- **weighted avg** - mean weighted by support (accounts for class imbalance)

Looking at the output above, class 1 (*Versicolor*) has lower recall because some of its samples were misclassified as class 2 (*Virginica*), visible as off-diagonal entries in the confusion matrix.